In [0]:
# ==========================================================
# PROYECTO TELCO - MACHINE LEARNING
# Paso 1: Exploración de datos
# ==========================================================

from pyspark.sql import functions as F

# Cargar datos limpios desde Silver
df_clientes = spark.table(
    "workspace.telco_silver.clientes"
).select(
    "cliente_id",
    "ciudad",
    "segmento",
    "fecha_alta",
    "estado"
)

# Total de clientes
print("=" * 55)
print("TOTAL DE CLIENTES:", df_clientes.count())
print("=" * 55)

# Estructura de las columnas
print("\nESTRUCTURA DE LOS DATOS")
df_clientes.printSchema()

# Distribución por estado
print("\nCLIENTES POR ESTADO")
df_clientes.groupBy(
    "estado"
).count().orderBy(
    "estado"
).show()

# Distribución por ciudad y segmento
print("\nCLIENTES POR CIUDAD Y SEGMENTO")
df_clientes.groupBy(
    "ciudad",
    "segmento"
).count().orderBy(
    "ciudad",
    "segmento"
).show()

# Fechas de alta
print("\nRANGO DE FECHAS DE ALTA")
df_clientes.agg(
    F.min("fecha_alta").alias("fecha_minima"),
    F.max("fecha_alta").alias("fecha_maxima")
).show(truncate=False)

In [0]:
# ==========================================================
# MACHINE LEARNING - INGENIERÍA DE CARACTERÍSTICAS
# ==========================================================

from pyspark.sql import functions as F

# Preparar variables del modelo
df_ml = (
    df_clientes
    .withColumn(
        "fecha_alta",
        F.to_date("fecha_alta")
    )
    .withColumn(
        "antiguedad_meses",
        F.floor(
            F.months_between(
                F.current_date(),
                F.col("fecha_alta")
            )
        )
    )
    .filter(
        F.col("ciudad").isNotNull()
        & F.col("segmento").isNotNull()
        & F.col("antiguedad_meses").isNotNull()
        & (F.col("antiguedad_meses") >= 0)
    )
    .select(
        "cliente_id",
        "ciudad",
        "segmento",
        "antiguedad_meses"
    )
)

# Validar cantidad de clientes
print("CLIENTES DISPONIBLES PARA EL MODELO:", df_ml.count())

# Analizar antigüedad por segmento
print("\nANTIGÜEDAD POR SEGMENTO")

df_ml.groupBy("segmento").agg(
    F.min("antiguedad_meses").alias("min_meses"),
    F.round(
        F.avg("antiguedad_meses"), 1
    ).alias("promedio_meses"),
    F.max("antiguedad_meses").alias("max_meses")
).show()

# Mostrar ejemplos
print("\nMUESTRA DE DATOS PARA MACHINE LEARNING")

display(df_ml.limit(10))

In [0]:
# ==========================================================
# MACHINE LEARNING - PREPARACIÓN DE CARACTERÍSTICAS
# ==========================================================

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)

# 1. Convertir ciudad y segmento a índices numéricos
indexador = StringIndexer(
    inputCols=["ciudad", "segmento"],
    outputCols=["ciudad_idx", "segmento_idx"],
    stringOrderType="alphabetAsc"
)

# 2. Aplicar One-Hot Encoding
codificador = OneHotEncoder(
    inputCols=["ciudad_idx", "segmento_idx"],
    outputCols=["ciudad_vec", "segmento_vec"],
    dropLast=False
)

# 3. Unir las variables en un vector
ensamblador = VectorAssembler(
    inputCols=[
        "ciudad_vec",
        "segmento_vec",
        "antiguedad_meses"
    ],
    outputCol="features_raw"
)

# 4. Escalar las características
escalador = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

# 5. Construir el pipeline de preparación
pipeline_features = Pipeline(
    stages=[
        indexador,
        codificador,
        ensamblador,
        escalador
    ]
)

# 6. Entrenar las transformaciones
modelo_features = pipeline_features.fit(df_ml)

# 7. Generar características para Machine Learning
df_features = modelo_features.transform(df_ml)

# 8. Validar resultados
print("CLIENTES PREPARADOS:", df_features.count())

print("\nCARACTERÍSTICAS GENERADAS")

df_features.select(
    "ciudad",
    "segmento",
    "antiguedad_meses",
    "features"
).show(10, truncate=False)

In [0]:
# ==========================================================
# MACHINE LEARNING - ENTRENAMIENTO K-MEANS
# ==========================================================

from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Evaluador de calidad de los grupos
evaluador = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

# Almacenar resultados y modelos
resultados = []
modelos = {}

# Probar diferentes cantidades de grupos
for k in [2, 3, 4, 5]:

    print(f"\nEntrenando K-Means con K={k}")

    kmeans = KMeans(
        k=k,
        seed=42,
        featuresCol="features",
        predictionCol="cluster",
        maxIter=25
    )

    # Entrenar modelo
    modelo = kmeans.fit(df_features)

    # Generar predicciones
    predicciones = modelo.transform(df_features)

    # Evaluar calidad
    silhouette = evaluador.evaluate(predicciones)

    # Guardar resultados
    resultados.append((k, float(silhouette)))
    modelos[k] = modelo

    print(f"Silhouette: {silhouette:.4f}")


# ==========================================================
# SELECCIONAR EL MEJOR K ENTRE LOS EVALUADOS
# ==========================================================

mejor_k, mejor_silhouette = max(
    resultados,
    key=lambda x: x[1]
)

print("\n" + "=" * 55)
print("RESULTADOS DEL ENTRENAMIENTO")
print("=" * 55)

for k, score in resultados:
    print(f"K={k} | Silhouette={score:.4f}")

print(f"\nMejor K evaluado: {mejor_k}")
print(f"Mejor Silhouette: {mejor_silhouette:.4f}")


# ==========================================================
# GENERAR SEGMENTACIÓN
# ==========================================================

mejor_modelo = modelos[mejor_k]

df_segmentado = mejor_modelo.transform(df_features)

print("\nDISTRIBUCIÓN DE CLIENTES POR CLUSTER")

df_segmentado.groupBy(
    "cluster"
).count().orderBy(
    "cluster"
).show()

In [0]:
# ==========================================================
# MACHINE LEARNING - PERFILAMIENTO DE CLUSTERS
# ==========================================================

from pyspark.sql import functions as F

# Resumen general de los clusters
perfil_clusters = df_segmentado.groupBy(
    "cluster"
).agg(
    F.count("*").alias("total_clientes"),

    F.round(
        F.avg("antiguedad_meses"), 1
    ).alias("antiguedad_promedio"),

    F.collect_set("ciudad").alias("ciudades"),

    F.collect_set("segmento").alias("segmentos")
).orderBy("cluster")

print("PERFIL GENERAL DE LOS CLUSTERS")
perfil_clusters.show(truncate=False)


# Distribución detallada
print("\nDISTRIBUCIÓN POR CLUSTER, CIUDAD Y SEGMENTO")

df_segmentado.groupBy(
    "cluster",
    "ciudad",
    "segmento"
).count().orderBy(
    "cluster",
    "ciudad",
    "segmento"
).show(50, truncate=False)

In [0]:
# ==========================================================
# MACHINE LEARNING - EVALUACIÓN ADICIONAL DE K
# ==========================================================

resultados_extra = []
modelos_extra = {}

for k in [6, 7, 8]:

    print(f"\nEntrenando K-Means con K={k}")

    kmeans = KMeans(
        k=k,
        seed=42,
        featuresCol="features",
        predictionCol="cluster",
        maxIter=25
    )

    modelo = kmeans.fit(df_features)

    predicciones = modelo.transform(df_features)

    silhouette = evaluador.evaluate(predicciones)

    resultados_extra.append((k, float(silhouette)))
    modelos_extra[k] = modelo

    print(f"Silhouette: {silhouette:.4f}")


# Consolidar resultados de K=2 hasta K=8
resultados_todos = sorted(
    resultados + resultados_extra,
    key=lambda x: x[0]
)

print("\n" + "=" * 55)
print("COMPARACIÓN FINAL DE MODELOS")
print("=" * 55)

for k, score in resultados_todos:
    print(f"K={k} | Silhouette={score:.4f}")


# Identificar el mejor K según Silhouette
mejor_k_global, mejor_score_global = max(
    resultados_todos,
    key=lambda x: x[1]
)

print("\nMEJOR RESULTADO DE LA EVALUACIÓN")

print(f"K: {mejor_k_global}")
print(f"Silhouette: {mejor_score_global:.4f}")

In [0]:
# ==========================================================
# MLFLOW - REGISTRO DE EXPERIMENTOS K-MEANS
# ==========================================================

import mlflow

# Total de clientes utilizados
total_clientes_ml = df_features.count()

# Registrar los resultados de los modelos ya evaluados
for k, score in resultados_todos:

    with mlflow.start_run(
        run_name=f"telco_kmeans_k{k}"
    ) as run:

        # Parámetros del modelo
        mlflow.log_param("algoritmo", "KMeans")
        mlflow.log_param("k", k)
        mlflow.log_param("seed", 42)
        mlflow.log_param("max_iter", 25)

        # Origen de datos
        mlflow.log_param(
            "tabla_origen",
            "workspace.telco_silver.clientes"
        )

        # Variables utilizadas
        mlflow.log_param(
            "variables",
            "ciudad,segmento,antiguedad_meses"
        )

        # Métricas
        mlflow.log_metric(
            "silhouette",
            float(score)
        )

        mlflow.log_metric(
            "total_clientes",
            total_clientes_ml
        )

        # Identificar el candidato seleccionado
        mlflow.set_tag(
            "modelo_seleccionado",
            "SI" if k == mejor_k_global else "NO"
        )

        print(
            f"✅ K={k} | "
            f"Silhouette={score:.4f} | "
            f"Run ID={run.info.run_id}"
        )

print("\n✅ REGISTRO DE EXPERIMENTOS FINALIZADO")

In [0]:
# ==========================================================
# MLFLOW - GUARDAR MODELO K-MEANS CON PREPROCESAMIENTO
# ==========================================================

import mlflow
import mlflow.spark

from pyspark.ml import PipelineModel

# 1. Combinar las transformaciones con K-Means
modelo_final = PipelineModel(
    stages=modelo_features.stages + [modelos[5]]
)

# 2. Verificar que el modelo completo funciona
df_prediccion = modelo_final.transform(df_ml)

total_predicciones = df_prediccion.count()

assert total_predicciones == 1000, (
    f"Se esperaban 1000 predicciones, "
    f"pero se obtuvieron {total_predicciones}"
)

print("✅ Modelo completo validado")
print(f"Clientes procesados: {total_predicciones}")

# 3. Identificador del experimento K=5
run_id_k5 = "3ecf204d49f6496dab452287808848db"

# 4. Guardar el modelo en la ejecución existente
# Guardar el modelo K-Means en MLflow
run_id_k5 = "3ecf204d49f6496dab452287808848db"

with mlflow.start_run(run_id=run_id_k5):

    info_modelo = mlflow.spark.log_model(
        spark_model=modelo_final,
        artifact_path="modelo_telco_kmeans_k5",
        dfs_tmpdir="/Volumes/workspace/telco_ml/mlflow_tmp"
    )

print("✅ MODELO GUARDADO EN MLFLOW")
print("URI del modelo:", info_modelo.model_uri)

In [0]:
# Crear esquema para los componentes de Machine Learning
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.telco_ml
""")

# Crear volumen administrado para MLflow
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.telco_ml.mlflow_tmp
""")

# Verificar que el volumen exista
display(
    spark.sql("SHOW VOLUMES IN workspace.telco_ml")
)

In [0]:
import mlflow
import mlflow.spark
from pyspark.sql import functions as F

# 1. Ruta del modelo guardado
model_uri = (
    "runs:/3ecf204d49f6496dab452287808848db/"
    "modelo_telco_kmeans_k5"
)

# 2. Recuperar el modelo desde MLflow
modelo_cargado = mlflow.spark.load_model(
    model_uri=model_uri,
    dfs_tmpdir="/Volumes/workspace/telco_ml/mlflow_tmp"
)

# 3. Aplicar el modelo a los clientes
df_validacion = modelo_cargado.transform(df_ml)

# 4. Validar resultados
total_clientes = df_validacion.count()
clientes_sin_cluster = df_validacion.filter(
    F.col("cluster").isNull()
).count()

assert total_clientes == 1000
assert clientes_sin_cluster == 0

print("✅ MODELO RECUPERADO CORRECTAMENTE")
print("Clientes procesados:", total_clientes)
print("Clientes sin cluster:", clientes_sin_cluster)

# 5. Mostrar distribución de los clusters
display(
    df_validacion
    .groupBy("cluster")
    .count()
    .orderBy("cluster")
)

In [0]:
from pyspark.sql import functions as F

# Tabla de destino
tabla_destino = "workspace.telco_ml.clientes_segmentados"

# Seleccionar resultados de la segmentación
df_resultado = (
    df_validacion
    .select(
        "cliente_id",
        "ciudad",
        "segmento",
        "antiguedad_meses",
        "cluster"
    )
    .withColumn("modelo_k", F.lit(5))
    .withColumn("fecha_segmentacion", F.current_timestamp())
)

# Crear tabla Delta sin sobrescribir datos existentes
# Crear tabla Delta en Unity Catalog
df_resultado.writeTo(tabla_destino).using("delta").create()

print("✅ TABLA CREADA CORRECTAMENTE")
print("Tabla:", tabla_destino)

# Validar datos guardados
display(
    spark.table(tabla_destino)
    .groupBy("cluster")
    .count()
    .orderBy("cluster")
)